# Deliverables 7–9 — Stability, noise boundary, and streaming compilation

Runs 50–1000-step off-equilibrium stability tests and native compilation across grid sizes/topologies. The full combined-circuit noise simulation is recorded as dependency-blocked because global lifted collision and streaming are not yet one compiled circuit.

In [1]:
from pathlib import Path
import sys
repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
import json, pandas as pd
from quantum_aero.advanced import collision_stability, compile_streaming_resources

stability=[]
for omega in (1.2,1.8,1.95,1.995):
 for speed in (.03,.1,.2,.3):
  for density in (-.02,0,.02):
   for steps in (50,200,1000): stability.append(collision_stability(omega,speed,density,steps))
stability_df=pd.DataFrame(stability); stability_df.to_csv(output_dir/"10_carleman_stability.csv",index=False)
stability_df.groupby(["omega","requested_steps"]).agg(stable_fraction=("stable","mean"),max_error=("relative_error","max"),min_population=("minimum_population","min"))

stable_fraction  max_error  min_population
omega requested_steps                                            
1.200 50                           1.0   0.004304        0.010948
      200                          1.0   0.004304        0.010948
      1000                         1.0   0.004304        0.010948
1.800 50                           1.0   0.004304        0.010948
      200                          1.0   0.004304        0.010948
      1000                         1.0   0.004304        0.010948
1.950 50                           1.0   0.003973        0.010966
      200                          1.0   0.004304        0.010948
      1000                         1.0   0.004304        0.010948
1.995 50                           1.0   0.000954        0.011131
      200                          1.0   0.002724        0.011034
      1000                         1.0   0.004275        0.010950

In [3]:
resources=[]
for n in (4,8,16,32,64,128):
 for topology in ("all_to_all","ring","line"):
  resources.append(compile_streaming_resources(n,topology))
resource_df=pd.DataFrame(resources); resource_df.to_csv(output_dir/"10_streaming_compilation.csv",index=False)
resource_df

,n,topology,logical_qubits,high_level_depth,high_level_size,native_depth,native_size,cx,u,clifford_t_proxy,compile_seconds,unused_direction_states
0,4,all_to_all,8,29,58,1416,1791,790,1001,110110,0.113642,7
1,4,ring,8,29,58,2615,3324,2323,1001,110110,0.157205,7
2,4,line,8,29,58,2948,3789,2788,1001,110110,0.236710,7
3,8,all_to_all,10,29,58,11102,15994,7376,8618,947980,0.848292,7
4,8,ring,10,29,58,21814,29842,21224,8618,947980,1.217761,7
5,8,line,10,29,58,23102,31711,23093,8618,947980,1.579802,7
6,16,all_to_all,12,29,58,30970,46388,21434,24954,2744940,2.082048,7
7,16,ring,12,29,58,66248,92528,67574,24954,2744940,3.625154,7
8,16,line,12,29,58,68341,96089,71135,24954,2744940,3.731683,7
9,32,all_to_all,14,29,58,96996,143549,66366,77183,8490130,6.751509,7


In [4]:
noise_status={"status":"blocked","reason":"a nontrivial global raw-f lifted collision+streaming circuit does not yet exist; applying noise to disconnected proxies would not answer the experiment",
"available_evidence":"07 executes the complete local collision circuit; 10 compiles complete streaming separately",
"next_action":"complete Q PREPARE/SELECT and global position-aware lift, then transpile and run Aer noise on that single circuit"}
(output_dir/"10_combined_noise_status.json").write_text(json.dumps(noise_status,indent=2))
assert len(stability_df)==144 and len(resource_df)==18
print("PASS: stability and streaming compilation completed. Combined noise honestly blocked by circuit integration.")

PASS: stability and streaming compilation completed. Combined noise honestly blocked by circuit integration.
